# Prepare hyperspectral imaging
Paired helicities across energy. Set raw/proc paths and verify scan metadata before running.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import h5py
import matplotlib.pyplot as plt
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "docs/source_manifest.json").is_file())
sys.path.insert(0, str(ROOT))
from library.mask_store import (
    load_red_mask_png,
    mask_png_paths,
    save_red_mask_png,
    save_mask_reference_png,
)
from scipy.ndimage import shift, binary_dilation, distance_transform_edt, map_coordinates
DATA_ROOT = Path("/path/to/2409_maxiv")  # Folder containing raw/ and proc/.
RAW_FOLDER = DATA_ROOT / "raw"
AVERAGE_FOLDER = DATA_ROOT / "proc"
MASK_ROOT = ROOT / "data/hyperspectral/masks"
SUPPORT_FILE = ROOT / "data/hyperspectral/support.h5"
CACHE = ROOT / "data/hyperspectral/prepared.h5"
FRAMES_PER_AVERAGE = 500  # Check against the acquisition metadata.

# (positive scan, negative scan, dark scan IDs, exposure seconds, state)
# Extend this list with all spectral pairs. Keep the same state for one energy scan.
PAIRS = [
    # (+ scan, - scan, dark scans, exposure seconds, state)
    (1152, 1150, (1151, 1153), 0.065, "state0"),
    (1155, 1157, (1156, 1158), 0.070, "state0"),
    (1162, 1160, (1159, 1161), 0.065, "state0"),
    (1167, 1170, (1169, 1171), 0.100, "state0"),
    (1175, 1173, (1172, 1174), 0.350, "state0"),
    (1178, 1180, (1177, 1181), 1.000, "state0"),
    (1185, 1183, (1177, 1181), 1.000, "state0"),
    (1191, 1189, (1187, 1190), 0.800, "state0"),
    (1196, 1194, (1193, 1195), 0.600, "state0"),
    #(1201, 1199, (1198, 1200), 0.300, "state0"),
    #(1229, 1227, (1226, 1228), 0.300, "state0"),
    (1204, 1206, (1205, 1203), 0.500, "state0"),
    (1235, 1232, (1231, 1234), 0.250, "state0"),
    (1241, 1239, (1237, 1238), 0.250, "state0"),
]
ENERGY_TOLERANCE_EV = 0.25
DETECTOR_CENTER = (1137, 1126)  # row, column
# Applies separately to every state, energy and opposite polarization.
BINNING = 2
CROP = 248  # Full-resolution detector pixels removed per edge before binning
USE_SAVED_PIXEL_MASK = True
RESCALE_HOLOGRAMS_WITH_ENERGY = True  # Set False to use the original detector grid
REFERENCE_ENERGY_EV = None  # None uses the median scan energy; or enter a positive eV value

RESCALE_INTERPOLATION_ORDER = 5  # Quintic spline interpolation (SciPy supports orders 0–5)

SATURATION = 0.75e4


## 1. Inspect acquisition and detector geometry

In [ ]:
def scan_snapshot(scan_id):
    path = RAW_FOLDER / f"2409_maxiv_{scan_id:05d}.h5"
    with h5py.File(path, "r") as handle:
        entry = handle[f"entry{scan_id}"]
        snapshot = entry["measurement/pre_scan_snapshot"]
        def scalar(name):
            if name not in snapshot:
                return None
            values = np.asarray(snapshot[name][()]).reshape(-1)
            if values.size == 0:
                return None
            if np.issubdtype(values.dtype, np.number):
                if not np.allclose(values, values[0]):
                    raise ValueError(f"Scan {scan_id}: {name} changes during acquisition")
                return float(values[0])
            return str(values[0])
        return {key: scalar(key) for key in ("energy", "helicity", "magOOP")}

if len(PAIRS) < 2:
    raise ValueError("Add at least two +/− energy pairs to PAIRS before running retrieval")
records = []
for pos_id, neg_id, dark_ids, exposure_s, state in PAIRS:
    pos_meta, neg_meta = scan_snapshot(pos_id), scan_snapshot(neg_id)
    ep, en = pos_meta["energy"], neg_meta["energy"]
    if ep is None or en is None or abs(ep - en) > ENERGY_TOLERANCE_EV:
        raise ValueError(f"Energy mismatch for {pos_id}/{neg_id}: {ep}, {en} eV")
    if pos_meta["magOOP"] is not None and neg_meta["magOOP"] is not None:
        if not np.isclose(pos_meta["magOOP"], neg_meta["magOOP"]):
            raise ValueError(f"Magnetic field changed within pair {pos_id}/{neg_id}")
    if exposure_s <= 0:
        raise ValueError(f"Invalid exposure for pair {pos_id}/{neg_id}")
    for scan_id, sign, meta in ((pos_id, +1, pos_meta), (neg_id, -1, neg_meta)):
        if not dark_ids:
            raise ValueError(f"Scan {scan_id} has no dark reference")
        records.append(dict(scan_id=int(scan_id), dark_ids=tuple(dark_ids), dark_id=int(dark_ids[0]),
                            polarization=sign, energy_eV=(ep + en) / 2,
                            state=str(state), exposure_s=float(exposure_s), metadata=meta))
energy_values = np.array(sorted({r["energy_eV"] for r in records}))
if len(energy_values) < 2:
    raise ValueError("The selected pairs contain fewer than two distinct energies")
print("Observations:", len(records), "energies:", energy_values)
for row in records:
    print(row["scan_id"], row["energy_eV"], row["polarization"],
          row["state"], row["dark_ids"], row["metadata"])

## 2. Correct, align, mask, and cache

In [ ]:
support_source = None
pixel_source = None
if SUPPORT_FILE.exists():
    with h5py.File(SUPPORT_FILE) as handle:
        if {"supportmask", "mask_pixel"}.issubset(handle.keys()):
            support_source = np.asarray(handle["supportmask"])
            pixel_source = np.asarray(handle["mask_pixel"]) != 0
if support_source is not None and support_source.ndim != 2:
    raise ValueError("Expected a 2D supportmask in the support HDF5 file")
if support_source is not None and pixel_source is not None and pixel_source.shape != support_source.shape:
    raise ValueError("Expected matching 2D supportmask and mask_pixel")
if not isinstance(BINNING, int) or BINNING < 1:
    raise ValueError("BINNING must be a positive integer")
with h5py.File(AVERAGE_FOLDER / f"scan_{records[0]['scan_id']:04d}_avg.h5") as handle:
    source_shape = np.asarray(handle['image']).squeeze().shape

support_reference_png, support_painted_png = mask_png_paths(MASK_ROOT, "supportmask")
pixel_reference_png, pixel_painted_png = mask_png_paths(MASK_ROOT, "mask_pixel")

if support_painted_png.exists():
    supportmask = load_red_mask_png(support_painted_png, expected_shape=source_shape).astype(np.uint8)
else:
    if support_source is None:
        raise FileNotFoundError("No reference support PNG exists and no support file was found. Create the supportmask PNG in data/hyperspectral/masks/.")
    support_preview = np.asarray(support_source, dtype=float)
    save_mask_reference_png(support_preview, support_reference_png, log_scale=False, colormap="viridis")
    supportmask = (support_source != 0).astype(np.uint8)
    save_red_mask_png(support_painted_png, supportmask, reference=support_reference_png)
    supportmask = load_red_mask_png(support_painted_png, expected_shape=source_shape).astype(np.uint8)

if pixel_painted_png.exists():
    pixel_source = load_red_mask_png(pixel_painted_png, expected_shape=source_shape) != 0
else:
    if pixel_source is None:
        raise FileNotFoundError("No detector PNG exists and no mask_pixel dataset was found. Create the mask_pixel PNG in data/hyperspectral/masks/.")
    reference_hologram = np.asarray(pixel_source, dtype=float)
    save_mask_reference_png(reference_hologram, pixel_reference_png, log_scale=False, colormap="viridis")
    save_red_mask_png(pixel_painted_png, pixel_source, reference=pixel_reference_png)
    pixel_source = load_red_mask_png(pixel_painted_png, expected_shape=source_shape) != 0

USE_SAVED_PIXEL_MASK = USE_SAVED_PIXEL_MASK and pixel_source is not None

def load_average(scan_id):
    with h5py.File(AVERAGE_FOLDER / f"scan_{scan_id:04d}_avg.h5", "r") as handle:
        image = np.asarray(handle["image"][()]).squeeze()
    if image.shape != source_shape:
        raise ValueError(f"Scan {scan_id}: shape {image.shape} != support {source_shape}")
    return image.astype(np.float32)

if RESCALE_INTERPOLATION_ORDER not in range(2, 6):
    raise ValueError("Use interpolation order 2–5 for energy registration")
reference_energy_eV = (float(np.median(energy_values)) if REFERENCE_ENERGY_EV is None
                       else float(REFERENCE_ENERGY_EV))
if not np.isfinite(reference_energy_eV) or reference_energy_eV <= 0:
    raise ValueError("REFERENCE_ENERGY_EV must be positive and finite")
centered_origin = np.asarray(source_shape, dtype=float) // 2
if RESCALE_HOLOGRAMS_WITH_ENERGY:
    grid_row, grid_col = np.indices(source_shape, dtype=np.float32)

def register_energy(image, excluded, energy_eV):
    """Resample a centered intensity and propagate its invalid-pixel mask."""
    scale = reference_energy_eV / float(energy_eV)
    if np.isclose(scale, 1.0, rtol=0, atol=1e-12):
        return image, excluded
    coords = (centered_origin[0] + scale * (grid_row - centered_origin[0]),
              centered_origin[1] + scale * (grid_col - centered_origin[1]))
    # Fill bad source pixels before the spline prefilter to avoid spreading their values.
    if np.any(excluded):
        if np.all(excluded):
            raise ValueError("No valid detector pixels available for registration")
        nearest = distance_transform_edt(excluded, return_distances=False,
                                         return_indices=True)
        clean = image.copy()
        clean[excluded] = image[tuple(index[excluded] for index in nearest)]
    else:
        clean = image
    registered = map_coordinates(clean, coords, order=RESCALE_INTERPOLATION_ORDER,
                                 mode="constant", cval=0.0, prefilter=True)
    # Mask the full spline footprint around invalid pixels as well as outside the image.
    footprint = RESCALE_INTERPOLATION_ORDER + 1
    unsafe = binary_dilation(excluded, iterations=footprint)
    warped_unsafe = map_coordinates(unsafe.astype(np.uint8), coords, order=0,
                                    mode="constant", cval=1) != 0
    return registered, warped_unsafe

dark_cache = {}
holograms, masks = [], []
for row in records:
    key = tuple(row['dark_ids'])
    if key not in dark_cache:
        dark_cache[key] = np.mean([load_average(i) for i in key], axis=0)
    raw = load_average(row['scan_id'])
    delta = np.asarray(source_shape)/2 - DETECTOR_CENTER
    corrected = shift(raw - dark_cache[key], delta, order=1, mode='constant', cval=0)
    excluded = shift((raw >= SATURATION).astype(np.uint8), delta, order=0,
                     mode='constant', cval=1) != 0
    if USE_SAVED_PIXEL_MASK:
        excluded |= pixel_source
    excluded |= ~np.isfinite(corrected)
    corrected = np.nan_to_num(corrected) / (FRAMES_PER_AVERAGE * row['exposure_s'])
    if RESCALE_HOLOGRAMS_WITH_ENERGY:
        corrected, excluded = register_energy(corrected, excluded, row['energy_eV'])
    holograms.append(corrected.astype(np.float32))
    masks.append(excluded.astype(np.uint8))

holograms, mask_pixel = np.stack(holograms), np.stack(masks)
CACHE.parent.mkdir(parents=True, exist_ok=True)
with h5py.File(CACHE, 'x') as out:
    for name, value in dict(holograms=holograms, mask_pixel=mask_pixel, supportmask=supportmask).items():
        out.create_dataset(name, data=value, compression='gzip')
    out.attrs['records_json'] = json.dumps(records)
    out.attrs['config_json'] = json.dumps(dict(
        crop=CROP, binning=BINNING, detector_center=DETECTOR_CENTER,
        reference_energy_ev=reference_energy_eV, rescale_with_energy=RESCALE_HOLOGRAMS_WITH_ENERGY,
        saturation=SATURATION, frames_per_average=FRAMES_PER_AVERAGE,
        dark_policy='mean of all configured dark averages', normalization='mean frame / exposure_s'))
print(CACHE)
